In [ ]:
# Import libraries
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models, expected_returns, plotting

# Data Wrangling

In [ ]:
# Define Dow 30 tickers
dow_30 = [
    'AAPL', 'AMGN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'DOW',
    'GS', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM',
    'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'V', 'VZ', 'WBA', 'WMT'
]

In [ ]:
# Download historical price data
data = yf.download(dow_30, start="2020-07-01", end="2024-12-31")['Close']
# Download DJIA index data
djia = yf.download('^DJI', start="2020-07-01", end="2024-12-31")['Close']
djia.name = 'DJIA'

In [ ]:
# # Check structure
# print(data.shape)
# print(data.describe())
# print(djia.shape)
# print(djia.describe())

In [ ]:
# Check Missing Values
missing_data = data.isnull().sum()
missing_djia = djia.isnull().sum()
print(missing_data[missing_data > 0])
print(missing_djia[missing_djia > 0])

# EDA

In [ ]:
# Calculate Daily Returns
returns = data.pct_change().dropna()
djia_returns = djia.pct_change().dropna()
# print(returns.head())
# print(djia_returns.head())

In [ ]:
summary = returns.describe().T[['mean', 'std', 'min', 'max']]
summary['mean (%)'] = summary['mean'] * 100
summary['std (%)'] = summary['std'] * 100
print(summary[['mean (%)', 'std (%)', 'min', 'max']])

In [ ]:
import plotly.express as px

# Box Plot of Closing Prices 
# Convert to long format for Plotly
price_long = data.reset_index().melt(id_vars='Date', var_name='Ticker', value_name='Price')

# Plot interactive box plot
fig = px.box(price_long, x='Ticker', y='Price', points='outliers', title="Box Plot of Dow 30 Prices (2020–2024)", color_discrete_sequence=['#3366CC'])

# Update layout
fig.update_layout(
    xaxis_title="Company Ticker",
    yaxis_title="Adjusted Closing Price (USD)",
    height=600,
    template="plotly_white"
)

fig.show()

In [ ]:
import plotly.express as px

# Compute correlation matrix
corr_matrix = returns.corr().round(2)

# Create heatmap
fig = px.imshow(
    corr_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu',
    zmin=-1,
    zmax=1,
    title='Correlation Matrix of Daily Returns'
)

fig.update_layout(
    width=900,
    height=850,
    xaxis_title="Ticker",
    yaxis_title="Ticker",
    template="plotly_white"
)

# Show interactive plot
fig.show()

In [ ]:
import plotly.graph_objects as go

# Combine and normalize
plot_data = data.copy()
plot_data['DJIA'] = djia
plot_data = plot_data.ffill()
normalized = plot_data / plot_data.iloc[0] * 100

# Plot with Plotly
fig = go.Figure()

# Dow 30 in grey
for ticker in data.columns:
    fig.add_trace(go.Scatter(
        x=normalized.index,
        y=normalized[ticker],
        mode='lines',
        line=dict(color='grey', width=1),
        name=ticker,
        opacity=0.4,
        hoverinfo='name+y',
        showlegend=False
    ))

# DJIA in blue
fig.add_trace(go.Scatter(
    x=normalized.index,
    y=normalized['DJIA'],
    mode='lines',
    line=dict(color='blue', width=3),
    name='DJIA'
))

# Layout
fig.update_layout(
    title="Normalized Price Trends: Dow 30 Stocks (Grey) vs DJIA (Blue)",
    xaxis_title="Date",
    yaxis_title="Normalized Price (Start = 100)",
    template="plotly_white",
    legend=dict(x=0.01, y=0.99),
    height=600
)

fig.show()

In [ ]:
import plotly.subplots as sp
import plotly.graph_objects as go
import math

tickers = returns.columns
n = len(tickers)
cols = 5
rows = math.ceil(n / cols)

# Create subplot grid
fig = sp.make_subplots(
    rows=rows, cols=cols,
    subplot_titles=tickers,
    horizontal_spacing=0.03,
    vertical_spacing=0.07
)

# Add histogram for each ticker
for i, ticker in enumerate(tickers):
    row = i // cols + 1
    col = i % cols + 1
    fig.add_trace(
        go.Histogram(x=returns[ticker], nbinsx=50, name=ticker, marker_color='skyblue', showlegend=False),
        row=row, col=col
    )

# Layout settings
fig.update_layout(
    height=rows * 250,
    width=1300,
    title_text="Histogram of Daily Returns for Dow 30 Stocks",
    template='plotly_white'
)

# Show interactive plot
fig.show()

# Portfolio Optimization

In [ ]:
# Calculate Expected Returns and Covariance Matrix
mu = expected_returns.mean_historical_return(data)
S = risk_models.sample_cov(data)

## Maximize Sharpe Ratio – No Weight Constraints
Not consider the risk-free rate yet.

In [ ]:
# Initialize Efficient Frontier
ef_unconstrained = EfficientFrontier(mu, S)

# Optimize for max Sharpe Ratio
weights_unconstrained = ef_unconstrained.max_sharpe()

# Clean and display weights
cleaned_weights_unconstrained = ef_unconstrained.clean_weights()
print("Portfolio Weights (Unconstrained):")
print(cleaned_weights_unconstrained)

# Show portfolio performance
perf_unconstrained = ef_unconstrained.portfolio_performance(verbose=True)

## Maximize Sharpe Ratio – With Max Weight Constraint (25%)

In [ ]:
# Initialize Efficient Frontier again (new object)
ef_constrained = EfficientFrontier(mu, S)

# Add max weight constraint
ef_constrained.add_constraint(lambda w: w <= 0.25)

# Optimize for max Sharpe Ratio
weights_constrained = ef_constrained.max_sharpe()

# Clean and display weights
cleaned_weights_constrained = ef_constrained.clean_weights()
print("Portfolio Weights (Constrained ≤ 25%):")
print(cleaned_weights_constrained)

# Show portfolio performance
perf_constrained = ef_constrained.portfolio_performance(verbose=True)

## Minimum Volatility – No Weight Constraints

In [ ]:
# Initialize Efficient Frontier
ef_minvol_unconstrained = EfficientFrontier(mu, S)

# Optimize for minimum volatility
weights_minvol_unconstrained = ef_minvol_unconstrained.min_volatility()

# Clean and print weights
cleaned_weights_minvol_unconstrained = ef_minvol_unconstrained.clean_weights()
print("Portfolio Weights (Unconstrained):")
print(cleaned_weights_minvol_unconstrained)

# Show performance
ef_minvol_unconstrained.portfolio_performance(verbose=True)

## Minimum Volatility – With Max Weight Constraint (25%)

In [ ]:
# Initialize Efficient Frontier
ef_minvol_constrained = EfficientFrontier(mu, S)

# Add max weight constraint (≤ 25%)
ef_minvol_constrained.add_constraint(lambda w: w <= 0.25)

# Optimize for minimum volatility
weights_minvol_constrained = ef_minvol_constrained.min_volatility()

# Clean and print weights
cleaned_weights_minvol_constrained = ef_minvol_constrained.clean_weights()
print("Portfolio Weights (Constrained ≤ 25%):")
print(cleaned_weights_minvol_constrained)

# Show performance
perf_minvol = ef_minvol_constrained.portfolio_performance(verbose=True)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Portfolio daily returns using cleaned weights
w_unconstrained = pd.Series(cleaned_weights_unconstrained).reindex(returns.columns).fillna(0)
w_constrained = pd.Series(cleaned_weights_constrained).reindex(returns.columns).fillna(0)
w_minvol = pd.Series(cleaned_weights_minvol_constrained).reindex(returns.columns).fillna(0)

portfolio_returns_unconstrained = returns.dot(w_unconstrained).squeeze()
portfolio_returns_constrained = returns.dot(w_constrained).squeeze()
portfolio_returns_minvol = returns.dot(w_minvol).squeeze()

# Compute cumulative returns
cum_djia = (1 + djia_returns).cumprod()
cum_unconstrained = (1 + portfolio_returns_unconstrained).cumprod()
cum_constrained = (1 + portfolio_returns_constrained).cumprod()
cum_minvol = (1 + portfolio_returns_minvol).cumprod()

# Combine and Plot
plt.figure(figsize=(12, 6))
plt.plot(cum_djia, label='DJIA (Index)', linewidth=2)
plt.plot(cum_unconstrained, label='Max Sharpe (Unconstrained)', linewidth=2)
plt.plot(cum_constrained, label='Max Sharpe (≤25%)', linewidth=2)
plt.plot(cum_minvol, label='Min Volatility (≤25%)', linewidth=2)

plt.title("Cumulative Daily Returns: Portfolios vs DJIA")
plt.xlabel("Date")
plt.ylabel("Cumulative Return (Base = 1)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Monte Carlo Simulation

## Best Fit Distributions for All Portfolios

In [ ]:
from fitter import Fitter

# Max Sharpe (Unconstrained)
Dist_unconstrained = Fitter(portfolio_returns_unconstrained.dropna())
Dist_unconstrained.fit()
print("Best fit for Max Sharpe (Unconstrained):")
Dist_unconstrained.summary()

In [ ]:
# Max Sharpe (Constrained ≤ 25%)
Dist_constrained = Fitter(portfolio_returns_constrained.dropna())
Dist_constrained.fit()
print("Best fit for Max Sharpe (Constrained ≤ 25%):")
Dist_constrained.summary()

In [ ]:
# Min Volatility
Dist_minvol = Fitter(portfolio_returns_minvol.dropna())
Dist_minvol.fit()
print("Best fit for Min Volatility:")
Dist_minvol.summary()

## Portfolio Value Simulation

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

# Monte Carlo Simulation Function
def run_value_simulation_from_fitter(fitter_obj, sims, days, initial_investment):
    best_dist_name = list(fitter_obj.get_best().keys())[0]
    best_params = fitter_obj.fitted_param[best_dist_name]
    
    dist = getattr(stats, best_dist_name)
    np.random.seed(24)
    daily_returns = dist.rvs(*best_params[:-2], loc=best_params[-2], scale=best_params[-1], size=(days, sims))
    price_paths = initial_investment * np.exp(np.cumsum(daily_returns, axis=0))
    return best_dist_name, best_params, price_paths

# Run Simulations for Each Portfolio
sims = 10000
days = 20
initial_investment = 100000

best_u, params_u, price_paths_u = run_value_simulation_from_fitter(Dist_unconstrained, sims, days, initial_investment)
best_c, params_c, price_paths_c = run_value_simulation_from_fitter(Dist_constrained, sims, days, initial_investment)
best_m, params_m, price_paths_m = run_value_simulation_from_fitter(Dist_minvol, sims, days, initial_investment)

# Plot Results
plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
plt.plot(price_paths_u, color='steelblue', alpha=0.05)
plt.title("Portfolio Value: Max Sharpe (Unconstrained)")
plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(price_paths_c, color='steelblue', alpha=0.05)
plt.title("Portfolio Value: Max Sharpe (≤ 25%)")
plt.xlabel("Days")
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(price_paths_m, color='steelblue', alpha=0.05)
plt.title("Portfolio Value: Min Volatility")
plt.xlabel("Days")
plt.grid(True)

plt.tight_layout()
plt.show()

# Simulation Summary
def summarize_simulation(matrix, ci_level=0.95):
    final_values = matrix[-1]
    mean_val = np.mean(final_values)
    std_val = np.std(final_values)
    lower_bound = np.percentile(final_values, (1 - ci_level) / 2 * 100)
    upper_bound = np.percentile(final_values, (1 + ci_level) / 2 * 100)
    return {
        'mean': mean_val,
        'std': std_val,
        f'{int(ci_level * 100)}% CI Lower': lower_bound,
        f'{int(ci_level * 100)}% CI Upper': upper_bound
    }

# Print Summary
summary_value_unconstrained = summarize_simulation(price_paths_u)
summary_value_constrained = summarize_simulation(price_paths_c)
summary_value_minvol = summarize_simulation(price_paths_m)

print("Max Sharpe (Unconstrained) – Simulated Portfolio Value:")
for k, v in summary_value_unconstrained.items():
    print(f"{k}: ${v:,.2f}")

print("\nMax Sharpe (Constrained ≤ 25%) – Simulated Portfolio Value:")
for k, v in summary_value_constrained.items():
    print(f"{k}: ${v:,.2f}")

print("\nMin Volatility (Unconstrained) – Simulated Portfolio Value:")
for k, v in summary_value_minvol.items():
    print(f"{k}: ${v:,.2f}")

## Portfolio Expected Return Simulation

In [ ]:
# Monte Carlo Simulation Function
def run_return_simulation_from_fitter(fitter_obj, sims, days):
    best_dist_name = list(fitter_obj.get_best().keys())[0]
    best_params = fitter_obj.fitted_param[best_dist_name]

    dist = getattr(stats, best_dist_name)
    np.random.seed(24)
    daily_returns = dist.rvs(*best_params[:-2], loc=best_params[-2], scale=best_params[-1], size=(days, sims))
    cumulative_returns = np.cumprod(1 + daily_returns, axis=0) - 1
    return best_dist_name, best_params, daily_returns, cumulative_returns

# Run Simulations for Each Portfolio
best_u_r, params_u_r, _, ret_u_cum = run_return_simulation_from_fitter(Dist_unconstrained, sims, days)
best_c_r, params_c_r, _, ret_c_cum = run_return_simulation_from_fitter(Dist_constrained, sims, days)
best_m_r, params_m_r, _, ret_m_cum = run_return_simulation_from_fitter(Dist_minvol, sims, days)

# Plot Results
plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
plt.plot(ret_u_cum, color='steelblue', alpha=0.05)
plt.title("Cumulative Return: Max Sharpe (Unconstrained)")
plt.xlabel("Days")
plt.ylabel("Cumulative Return")
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(ret_c_cum, color='steelblue', alpha=0.05)
plt.title("Cumulative Return: Max Sharpe (≤ 25%)")
plt.xlabel("Days")
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(ret_m_cum, color='steelblue', alpha=0.05)
plt.title("Cumulative Return: Min Volatility")
plt.xlabel("Days")
plt.grid(True)

plt.tight_layout()
plt.show()

# Simulation Summary
summary_return_unconstrained = summarize_simulation(ret_u_cum)
summary_return_constrained = summarize_simulation(ret_c_cum)
summary_return_minvol = summarize_simulation(ret_m_cum)

# Print Summary
print("\nMax Sharpe (Unconstrained) – Simulated Return:")
for k, v in summary_return_unconstrained.items():
    print(f"{k}: {v:.2%}")

print("\nMax Sharpe (Constrained ≤ 25%) – Simulated Return:")
for k, v in summary_return_constrained.items():
    print(f"{k}: {v:.2%}")

print("\nMin Volatility (Unconstrained) – Simulated Return:")
for k, v in summary_return_minvol.items():
    print(f"{k}: {v:.2%}")